In [1]:
from openai import OpenAI
import os
import pandas as pd
from pynvml import *
import time

In [2]:
nvmlInit()
handle = nvmlDeviceGetHandleByIndex(0)

# GPU model name
name = nvmlDeviceGetName(handle)
name = name.replace("NVIDIA GeForce ","")

# Memory info
mem_info = nvmlDeviceGetMemoryInfo(handle)
total_vram_gb = mem_info.total / (1024**3)

# GPU model identifier in the perf tables
gpu_model = f"{name} {total_vram_gb:.0f}GB"

print(f"Testing performance on the following GPU: {gpu_model}")

Testing performance on the following GPU: RTX 5090 32GB


In [3]:
def get_vram_used():
    info = nvmlDeviceGetMemoryInfo(handle)
    return info.used / 10**9

In [4]:
initial_vram = get_vram_used()
print(f"Initial VRAM used: {initial_vram:.3f}GB")

Initial VRAM used: 26.806GB


### llama.cpp config

https://github.com/ggml-org/llama.cpp/blob/master/docs/docker.md

```bash
docker run --runtime nvidia --gpus all -v $WORDSLAB_MODELS/huggingface:/root/.cache/huggingface --env "HF_TOKEN=$HF_TOKEN" -p 8000:8000  ghcr.io/ggml-org/llama.cpp:server-cuda13-b8757 -hf unsloth/gemma-4-26B-A4B-it-GGUF:UD-Q4_K_XL --alias "gemma-4-26B-A4B" -n 512 --temp 1.0 --top-p 0.95 --top-k 64  --chat-template-kwargs '{"enable_thinking":true}' --port 8000 --host 0.0.0.0 
```

https://github.com/ggml-org/llama.cpp/blob/master/tools/server/README.md

-c, --ctx-size N	size of the prompt context (default: 0, 0 = loaded from model)
(env: LLAMA_ARG_CTX_SIZE)

-n, --predict, --n-predict N	number of tokens to predict (default: -1, -1 = infinity)
(env: LLAMA_ARG_N_PREDICT)

-b, --batch-size N	logical maximum batch size (default: 2048)
(env: LLAMA_ARG_BATCH)

-ub, --ubatch-size N	physical maximum batch size (default: 512)
(env: LLAMA_ARG_UBATCH)

-fa, --flash-attn [on|off|auto]	set Flash Attention use ('on', 'off', or 'auto', default: 'auto')
(env: LLAMA_ARG_FLASH_ATTN)

--perf, --no-perf	whether to enable internal libllama performance timings (default: false)
(env: LLAMA_ARG_PERF)

-kvo, --kv-offload, -nkvo, --no-kv-offload	whether to enable KV cache offloading (default: enabled)
(env: LLAMA_ARG_KV_OFFLOAD)

-ctk, --cache-type-k TYPE	KV cache data type for K
allowed values: f32, f16, bf16, q8_0, q4_0, q4_1, iq4_nl, q5_0, q5_1
(default: f16)
(env: LLAMA_ARG_CACHE_TYPE_K)

-ctv, --cache-type-v TYPE	KV cache data type for V
allowed values: f32, f16, bf16, q8_0, q4_0, q4_1, iq4_nl, q5_0, q5_1
(default: f16)
(env: LLAMA_ARG_CACHE_TYPE_V)

-ngl, --gpu-layers, --n-gpu-layers N	max. number of layers to store in VRAM, either an exact number, 'auto', or 'all' (default: auto)
(env: LLAMA_ARG_N_GPU_LAYERS)

-fit, --fit [on|off]	whether to adjust unset arguments to fit in device memory ('on' or 'off', default: 'on')
(env: LLAMA_ARG_FIT)

--warmup, --no-warmup	whether to perform warmup with an empty run (default: enabled)

--cache-prompt, --no-cache-prompt	whether to enable prompt caching (default: enabled)
(env: LLAMA_ARG_CACHE_PROMPT)

--cache-reuse N	min chunk size to attempt reusing from the cache via KV shifting, requires prompt caching to be enabled (default: 0)
(card)
(env: LLAMA_ARG_CACHE_REUSE)

--metrics	enable prometheus compatible metrics endpoint (default: disabled)
(env: LLAMA_ARG_ENDPOINT_METRICS)

https://unsloth.ai/docs/models/qwen3.5

https://unsloth.ai/docs/models/gemma-4

> docker pull ghcr.io/ggml-org/llama.cpp:server-cuda13-b8757

> pip install huggingface_hub hf_transfer

> hf download unsloth/gemma-4-26B-A4B-it-GGUF --include "*mmproj-BF16*" --include "*UD-Q4_K_XL*"

> hf download unsloth/gemma-4-31B-it-GGUF --include "*mmproj-BF16*" --include "*UD-Q4_K_XL*"

> hf download unsloth/Qwen3.5-27B-GGUF --include "*mmproj-F16*" --include "*UD-Q4_K_XL*" 

> hf download unsloth/Qwen3.5-35B-A3B-GGUF --include "*mmproj-F16*" --include "*UD-Q4_K_XL*" 

unsloth/gemma-4-26B-A4B-it-GGUF:UD-Q4_K_XL

```bash
docker run --runtime nvidia --gpus all -v $WORDSLAB_MODELS/huggingface:/root/.cache/huggingface --env "HF_TOKEN=$HF_TOKEN" -p 8000:8000  ghcr.io/ggml-org/llama.cpp:server-cuda13-b8757 -hf unsloth/gemma-4-26B-A4B-it-GGUF:UD-Q4_K_XL --alias "gemma-4-26B-A4B" -n 512 --temp 1.0 --top-p 0.95 --top-k 64  --chat-template-kwargs '{"enable_thinking":true}' --port 8000 --host 0.0.0.0
```


unsloth/gemma-4-26B-A4B-it-GGUF:UD-Q4_K_XL => llama_kv_cache: size = 5120.00 MiB (262144 cells,   5 layers,  4/1 seqs), K (f16): 2560.00 MiB, V (f16): 2560.00 MiB

unsloth/gemma-4-31B-it-GGUF:UD-Q4_K_XL => llama_kv_cache: size = 8000.00 MiB (102400 cells,  10 layers,  4/1 seqs), K (f16): 4000.00 MiB, V (f16): 4000.00 MiB

unsloth/Qwen3.5-27B-GGUF:UD-Q4_K_XL => llama_kv_cache: size = 11856.00 MiB (189696 cells,  16 layers,  4/1 seqs), K (f16): 5928.00 MiB, V (f16): 5928.00 MiB

unsloth/Qwen3.5-35B-A3B-GGUF:UD-Q4_K_XL => llama_kv_cache: size = 5120.00 MiB (262144 cells,  10 layers,  4/1 seqs), K (f16): 2560.00 MiB, V (f16): 2560.00 MiB

In [5]:
client = OpenAI(base_url="http://127.0.0.1:8000/v1",api_key="")

In [37]:
model = "Qwen3.5-35B-A3B"
max_context_length = 262144

In [38]:
def compute_prompt_tokens(prompt):
    cc = client.chat.completions.create(model=model, messages = [{'role': 'user', 'content': prompt}], max_tokens = 1)
    return cc.usage.prompt_tokens

In [39]:
def measure_perfs(prompt_repeat):
    start = time.perf_counter()
    stream = client.chat.completions.create(model=model, messages = [{'role': 'user', 'content': (base_text*prompt_repeat)}], max_tokens = 100, stream=True, stream_options={"include_usage": True})
    first_token_time = None
    for chunk in stream:
        if first_token_time is None:
            first_token_time = time.perf_counter()
            ttf = first_token_time - start
        # capture usage if present (usually only in final chunk)
        if hasattr(chunk, "usage") and chunk.usage:
            usage = chunk.usage
    end = time.perf_counter()
    total = end - start
    return usage.prompt_tokens,usage.prompt_tokens/ttf,usage.completion_tokens,usage.completion_tokens/total
#    return response.prompt_eval_count,response.prompt_eval_count/(response.prompt_eval_duration/10**9), response.eval_count,response.eval_count/(response.eval_duration/10**9)

In [40]:
context_sizes = [1024*p for p in range(1,8)] + [8192*p for p in range(1,17)] + [16384*p for p in range(9,17)] + [32768*p for p in range(9,17)] + [65536*p for p in range(9,17)]

In [41]:
ctx1k = context_sizes[0]
ctx2k = context_sizes[1]
ctx4k = context_sizes[3]
ctx8k = context_sizes[7]
ctx16k = context_sizes[8]
ctx32k = context_sizes[10]
ctx64k = context_sizes[14]
ctx128k = context_sizes[22]
ctx192k = context_sizes[26]
ctx256k = context_sizes[30]
ctx384k = context_sizes[34]
ctx512k = context_sizes[38]
ctx1M = context_sizes[46]

In [42]:
print(",".join([f"{int(context_sizes[ctx_index]/1024)}k" for ctx_index in range(7,35)]))
print(",".join([f"{context_sizes[ctx_index]}" for ctx_index in range(7,35)]))

8k,16k,24k,32k,40k,48k,56k,64k,72k,80k,88k,96k,104k,112k,120k,128k,144k,160k,176k,192k,208k,224k,240k,256k,288k,320k,352k,384k
8192,16384,24576,32768,40960,49152,57344,65536,73728,81920,90112,98304,106496,114688,122880,131072,147456,163840,180224,196608,212992,229376,245760,262144,294912,327680,360448,393216


In [43]:
base_text = '\nAbsolutely! Let’s unpack this carefully and thoroughly. The concept of a **token** comes up most often in natural language processing (NLP), programming, and blockchain/crypto, but I’ll focus first on the context most relevant to AI models like me, and then touch briefly on other uses for clarity.\n\n---\n\n## **1. Tokens in AI and NLP (Language Models)**\n\nIn the context of language models like GPT:\n\n**A token is essentially a piece of text that the model treats as a unit for processing.**\n\nThink of it as a “chunk” of text, which could be:\n\n* A word (“apple” → 1 token)\n* Part of a word (“unbelievable” → might be split into “un”, “believ”, “able” → 3 tokens)\n* A single character or punctuation mark ("," or "!" → 1 token)\n* Even whitespace in some systems.\n\nLanguage models **do not process text character by character or word by word** in the strict sense; they process these tokens. This is because using tokens allows the model to handle text efficiently while capturing meaningful patterns.\n\n### **Why not just words?**\n\nWords are tricky because:\n\n* Some words are very long and rare.\n* Some languages (like Chinese, Japanese) don’t use spaces to separate words.\n* Using subword tokens balances between vocabulary size and representation power.\n\nBy using tokens, a model can efficiently represent language while keeping the total vocabulary manageable.\n\n---\n\n### **Tokenization: How text becomes tokens**\n\nThe process of breaking text into tokens is called **tokenization**. There are different tokenization methods:\n\n1. **Whitespace tokenization**\n\n   * Splits text at spaces.\n   * `"I love pizza!"` → `["I", "love", "pizza!"]`\n   * Simple, but can miss nuances like punctuation.\n\n2. **Subword tokenization (most modern models use this)**\n\n   * Breaks rare words into smaller pieces.\n   * `"unbelievable"` → `["un", "believ", "able"]`\n   * Helps the model understand new words it hasn\'t seen during training.\n\n3. **Character-level tokenization**\n\n   * Treats each character as a token.\n   * `"cat"` → `["c", "a", "t"]`\n   * Rarely used alone because sequences become very long.\n\n4. **Byte Pair Encoding (BPE)**\n\n   * A popular method used in GPT models.\n   * Starts with characters and iteratively merges common sequences to form subwords.\n   * Efficiently handles rare words while keeping the vocabulary compact.\n\n---\n\n### **Example of token counts**\n\nLet’s take the sentence:\n\n```\nHello, world! I love programming.\n```\n\nA GPT-style tokenizer might break it into tokens like:\n\n```\n["Hello", ",", " world", "!", " I", " love", " program", "ming", "."]\n```\n\nHere, it’s **9 tokens**. Notice:\n\n* Punctuation is separate.\n* Spaces before words are included as part of the token in some tokenizers.\n* Words like “programming” are split.\n\n---\n\n### **Why tokens matter in language models**\n\n1. **Input length**\n\n   * Models have a limit on the number of tokens they can process at once (e.g., GPT-4’s 8,192 or 32k token limit).\n   * Longer text → more tokens → may exceed the model’s limit.\n\n2. **Cost & computation**\n\n   * In APIs like OpenAI’s, you pay per token.\n   * More tokens → more computation → higher cost.\n\n3. **Context & memory**\n\n   * Models can only “remember” information within a token limit.\n   * Efficient tokenization allows models to fit more meaningful text into memory.\n\n---\n\n## **2. Tokens in Programming**\n\nA token in programming is a **syntactic unit of code**:\n\n* Keywords (`if`, `for`, `return`)\n* Identifiers (`myVariable`, `functionName`)\n* Operators (`+`, `-`, `=`)\n* Literals (`42`, `"hello"`)\n* Punctuation (`;`, `{`, `}`)\n\n**Example:**\n\n```python\nx = 42 + 7\n```\n\nTokenization would produce:\n\n```\n["x", "=", "42", "+", "7"]\n```\n\nThis is similar in principle: breaking a stream of characters into meaningful chunks the compiler or interpreter can understand.\n\n---\n\n## **3. Tokens in Blockchain / Crypto**\n\nHere, a token is a **digital asset** issued on a blockchain, representing something of value:\n\n* **Utility tokens** – give access to a service (e.g., Binance Coin for exchange fees).\n* **Security tokens** – represent ownership or investment.\n* **NFTs (non-fungible tokens)** – unique digital items (art, collectibles).\n\nThis is different from NLP, but the common theme is that a token is a **unit of something meaningful**, whether text, code, or digital value.\n\n---\n\n## **Key Takeaways**\n\n* **In AI/NLP:** Token = smallest meaningful piece of text for a model to process.\n* **In programming:** Token = smallest meaningful element of code.\n* **In crypto:** Token = a digital representation of value or rights.\n\n**Analogy:** Tokens are like LEGO blocks. Each block may be a full piece (word) or a part of a bigger structure (subword). You can build complex things (sentences, programs, ecosystems) from these basic building blocks.\n\n---\n\nIf you want, I can make a **visual diagram showing how a sentence gets broken into tokens and why that matters for GPT**, which makes this concept much easier to grasp.\n\nDo you want me to do that?\n'

In [44]:
# Load the model and tokenize the test text
base_tokens = compute_prompt_tokens(base_text)   

# Test at all prompt sizes until context_length - 100 (output tokens)
previous_prompt_tokens = -1
for prompt_size in context_sizes:
    prompt_repeat = int(prompt_size/base_tokens)
    prompt_tokens = prompt_repeat * base_tokens
    if prompt_tokens > (max_context_length - 100):
        break
    if prompt_tokens != previous_prompt_tokens:
        previous_prompt_tokens = prompt_tokens
        start_time = time.time()
        real_prompt_tokens,prompt_tokens_per_sec, output_tokens,output_tokens_per_sec = measure_perfs(prompt_repeat)
        end_time = time.time()
        elapsed_time = end_time - start_time
        print(f"{gpu_model},{model},{real_prompt_tokens},{int(prompt_tokens_per_sec)},{output_tokens_per_sec:.2f}")
        if elapsed_time > 60:
            break

RTX 5090 32GB,Qwen3.5-35B-A3B,10,49,128.04
RTX 5090 32GB,Qwen3.5-35B-A3B,1271,4167,110.69
RTX 5090 32GB,Qwen3.5-35B-A3B,2533,6351,100.37
RTX 5090 32GB,Qwen3.5-35B-A3B,3795,9870,102.05
RTX 5090 32GB,Qwen3.5-35B-A3B,5057,12338,99.05
RTX 5090 32GB,Qwen3.5-35B-A3B,6319,15714,100.64
RTX 5090 32GB,Qwen3.5-35B-A3B,7581,18246,102.21
RTX 5090 32GB,Qwen3.5-35B-A3B,15153,11245,51.60
RTX 5090 32GB,Qwen3.5-35B-A3B,23987,14277,43.26
RTX 5090 32GB,Qwen3.5-35B-A3B,31559,21730,48.62
RTX 5090 32GB,Qwen3.5-35B-A3B,40393,22394,40.82
RTX 5090 32GB,Qwen3.5-35B-A3B,47965,29891,44.62
RTX 5090 32GB,Qwen3.5-35B-A3B,56799,28881,37.57
RTX 5090 32GB,Qwen3.5-35B-A3B,64371,37559,42.41
RTX 5090 32GB,Qwen3.5-35B-A3B,73205,34226,35.27
RTX 5090 32GB,Qwen3.5-35B-A3B,80777,42682,38.70
RTX 5090 32GB,Qwen3.5-35B-A3B,88349,44736,37.56
RTX 5090 32GB,Qwen3.5-35B-A3B,97183,40595,32.05
RTX 5090 32GB,Qwen3.5-35B-A3B,104755,48618,34.79
RTX 5090 32GB,Qwen3.5-35B-A3B,113589,43721,29.95
RTX 5090 32GB,Qwen3.5-35B-A3B,121161,52462,32.8

unsloth/gemma-4-26B-A4B-it-GGUF:UD-Q4_K_XL

```bash
docker run --runtime nvidia --gpus all -v $WORDSLAB_MODELS/huggingface:/root/.cache/huggingface --env "HF_TOKEN=$HF_TOKEN" -p 8000:8000  ghcr.io/ggml-org/llama.cpp:server-cuda13-b8757 -hf unsloth/gemma-4-26B-A4B-it-GGUF:UD-Q4_K_XL --alias "gemma-4-26B-A4B" --temp 1.0 --top-p 0.95 --top-k 64  --chat-template-kwargs '{"enable_thinking":true}' --port 8000 --host 0.0.0.0
```

```
RTX 5090 32GB,gemma-4-26B-A4B,16,28,88.13
RTX 5090 32GB,gemma-4-26B-A4B,1274,13953,145.74
RTX 5090 32GB,gemma-4-26B-A4B,2533,3866,76.01
RTX 5090 32GB,gemma-4-26B-A4B,3792,3335,55.05
RTX 5090 32GB,gemma-4-26B-A4B,5051,3829,50.36
RTX 5090 32GB,gemma-4-26B-A4B,6310,4402,47.13
RTX 5090 32GB,gemma-4-26B-A4B,7569,4777,44.89
RTX 5090 32GB,gemma-4-26B-A4B,15123,6765,34.68
RTX 5090 32GB,gemma-4-26B-A4B,23936,9620,31.75
RTX 5090 32GB,gemma-4-26B-A4B,31490,12820,31.82
RTX 5090 32GB,gemma-4-26B-A4B,40303,14361,28.57
RTX 5090 32GB,gemma-4-26B-A4B,47857,17653,29.17
RTX 5090 32GB,gemma-4-26B-A4B,56670,18411,26.43
RTX 5090 32GB,gemma-4-26B-A4B,64224,21800,27.04
RTX 5090 32GB,gemma-4-26B-A4B,71778,23973,26.71
RTX 5090 32GB,gemma-4-26B-A4B,80591,23187,23.59
RTX 5090 32GB,gemma-4-26B-A4B,88145,27200,24.96
RTX 5090 32GB,gemma-4-26B-A4B,96958,26282,22.19
RTX 5090 32GB,gemma-4-26B-A4B,104512,30331,23.55
RTX 5090 32GB,gemma-4-26B-A4B,113325,29282,21.41
RTX 5090 32GB,gemma-4-26B-A4B,120879,33296,22.41
RTX 5090 32GB,gemma-4-26B-A4B,128433,35056,22.20
RTX 5090 32GB,gemma-4-26B-A4B,144800,19924,12.30
RTX 5090 32GB,gemma-4-26B-A4B,161167,20725,11.51
RTX 5090 32GB,gemma-4-26B-A4B,177534,21339,10.82
RTX 5090 32GB,gemma-4-26B-A4B,193901,22473,10.46
RTX 5090 32GB,gemma-4-26B-A4B,210268,23170,9.96
RTX 5090 32GB,gemma-4-26B-A4B,226635,23787,9.51
RTX 5090 32GB,gemma-4-26B-A4B,241743,25638,9.57
RTX 5090 32GB,gemma-4-26B-A4B,258110,24866,8.77
```

unsloth/gemma-4-31B-it-GGUF:UD-Q4_K_XL

```bash
docker run --runtime nvidia --gpus all -v $WORDSLAB_MODELS/huggingface:/root/.cache/huggingface --env "HF_TOKEN=$HF_TOKEN" -p 8000:8000  ghcr.io/ggml-org/llama.cpp:server-cuda13-b8757 -hf unsloth/gemma-4-31B-it-GGUF:UD-Q4_K_XL --alias "gemma-4-31B" --temp 1.0 --top-p 0.95 --top-k 64  --chat-template-kwargs '{"enable_thinking":true}' --port 8000 --host 0.0.0.0
```

```
RTX 5090 32GB,gemma-4-31B,16,8,28.58
RTX 5090 32GB,gemma-4-31B,1274,4720,52.20
RTX 5090 32GB,gemma-4-31B,2533,1078,24.65
RTX 5090 32GB,gemma-4-31B,3792,894,16.56
RTX 5090 32GB,gemma-4-31B,5051,1026,14.69
RTX 5090 32GB,gemma-4-31B,6310,1145,13.51
RTX 5090 32GB,gemma-4-31B,7569,1316,13.12
RTX 5090 32GB,gemma-4-31B,15123,1915,10.17
RTX 5090 32GB,gemma-4-31B,23936,2759,9.39
RTX 5090 32GB,gemma-4-31B,31490,1117,3.31
```

unsloth/Qwen3.5-27B-GGUF:UD-Q4_K_XL 

```bash
docker run --runtime nvidia --gpus all -v $WORDSLAB_MODELS/huggingface:/root/.cache/huggingface --env "HF_TOKEN=$HF_TOKEN" -p 8000:8000  ghcr.io/ggml-org/llama.cpp:server-cuda13-b8757 -hf unsloth/Qwen3.5-27B-GGUF:UD-Q4_K_XL --alias "Qwen3.5-27B" --temp 0.6 --top-p 0.95 --top-k 20 --min-p 0.00 --chat-template-kwargs '{"enable_thinking":true}' --port 8000 --host 0.0.0.0
```

```
RTX 5090 32GB,Qwen3.5-27B,10,23,19.04
RTX 5090 32GB,Qwen3.5-27B,1271,2347,48.25
RTX 5090 32GB,Qwen3.5-27B,2533,3641,44.21
RTX 5090 32GB,Qwen3.5-27B,3795,5290,43.75
RTX 5090 32GB,Qwen3.5-27B,5057,7038,43.49
RTX 5090 32GB,Qwen3.5-27B,6319,8799,43.86
RTX 5090 32GB,Qwen3.5-27B,7581,10035,43.00
RTX 5090 32GB,Qwen3.5-27B,15153,5937,24.09
RTX 5090 32GB,Qwen3.5-27B,23987,7299,20.31
RTX 5090 32GB,Qwen3.5-27B,31559,10523,21.49
RTX 5090 32GB,Qwen3.5-27B,40393,10472,18.00
RTX 5090 32GB,Qwen3.5-27B,47965,13419,18.88
RTX 5090 32GB,Qwen3.5-27B,56799,12237,15.58
RTX 5090 32GB,Qwen3.5-27B,64371,14926,16.39
RTX 5090 32GB,Qwen3.5-27B,73205,13108,13.48
RTX 5090 32GB,Qwen3.5-27B,80777,15880,14.42
RTX 5090 32GB,Qwen3.5-27B,88349,16172,13.53
RTX 5090 32GB,Qwen3.5-27B,97183,14222,11.41
RTX 5090 32GB,Qwen3.5-27B,104755,17092,12.36
RTX 5090 32GB,Qwen3.5-27B,113589,14814,10.33
RTX 5090 32GB,Qwen3.5-27B,121161,17664,11.26
RTX 5090 32GB,Qwen3.5-27B,129995,15347,9.49
RTX 5090 32GB,Qwen3.5-27B,146401,9119,5.49
RTX 5090 32GB,Qwen3.5-27B,161545,10045,5.47
RTX 5090 32GB,Qwen3.5-27B,177951,9473,4.75
```

unsloth/Qwen3.5-35B-A3B-GGUF:UD-Q4_K_XL

```bash
docker run --runtime nvidia --gpus all -v $WORDSLAB_MODELS/huggingface:/root/.cache/huggingface --env "HF_TOKEN=$HF_TOKEN" -p 8000:8000  ghcr.io/ggml-org/llama.cpp:server-cuda13-b8757 -hf unsloth/Qwen3.5-35B-A3B-GGUF:UD-Q4_K_XL --alias "Qwen3.5--35B-A3B" --temp 0.6 --top-p 0.95 --top-k 20 --min-p 0.00 --chat-template-kwargs '{"enable_thinking":true}' --port 8000 --host 0.0.0.0
```

```
RTX 5090 32GB,Qwen3.5-35B-A3B,10,49,128.04
RTX 5090 32GB,Qwen3.5-35B-A3B,1271,4167,110.69
RTX 5090 32GB,Qwen3.5-35B-A3B,2533,6351,100.37
RTX 5090 32GB,Qwen3.5-35B-A3B,3795,9870,102.05
RTX 5090 32GB,Qwen3.5-35B-A3B,5057,12338,99.05
RTX 5090 32GB,Qwen3.5-35B-A3B,6319,15714,100.64
RTX 5090 32GB,Qwen3.5-35B-A3B,7581,18246,102.21
RTX 5090 32GB,Qwen3.5-35B-A3B,15153,11245,51.60
RTX 5090 32GB,Qwen3.5-35B-A3B,23987,14277,43.26
RTX 5090 32GB,Qwen3.5-35B-A3B,31559,21730,48.62
RTX 5090 32GB,Qwen3.5-35B-A3B,40393,22394,40.82
RTX 5090 32GB,Qwen3.5-35B-A3B,47965,29891,44.62
RTX 5090 32GB,Qwen3.5-35B-A3B,56799,28881,37.57
RTX 5090 32GB,Qwen3.5-35B-A3B,64371,37559,42.41
RTX 5090 32GB,Qwen3.5-35B-A3B,73205,34226,35.27
RTX 5090 32GB,Qwen3.5-35B-A3B,80777,42682,38.70
RTX 5090 32GB,Qwen3.5-35B-A3B,88349,44736,37.56
RTX 5090 32GB,Qwen3.5-35B-A3B,97183,40595,32.05
RTX 5090 32GB,Qwen3.5-35B-A3B,104755,48618,34.79
RTX 5090 32GB,Qwen3.5-35B-A3B,113589,43721,29.95
RTX 5090 32GB,Qwen3.5-35B-A3B,121161,52462,32.84
RTX 5090 32GB,Qwen3.5-35B-A3B,129995,45987,27.71
RTX 5090 32GB,Qwen3.5-35B-A3B,146401,28577,16.94
RTX 5090 32GB,Qwen3.5-35B-A3B,161545,31818,17.00
RTX 5090 32GB,Qwen3.5-35B-A3B,177951,30555,15.02
RTX 5090 32GB,Qwen3.5-35B-A3B,194357,31122,14.09
RTX 5090 32GB,Qwen3.5-35B-A3B,210763,31497,13.20
RTX 5090 32GB,Qwen3.5-35B-A3B,227169,32387,12.61
RTX 5090 32GB,Qwen3.5-35B-A3B,243575,33002,12.02
RTX 5090 32GB,Qwen3.5-35B-A3B,259981,33168,11.38
```